In [ ]:
import openpyxlimport pandas as pdimport numpy as npfrom torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSamplerfrom tqdm import tqdmdata=pd.read_excel('./data/datos.xlsx')data['parsed'] = data['parsed'].astype(str)data['parsed'] = data['parsed'].str.split(r'\$\%\&') #separar las frases con los separadoresdata['text'] = data['text'].str.strip() #eliminar espacios rarosdata

In [ ]:
data = data.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
from sklearn.model_selection import train_test_split# Assuming you have your data in X (text) and y (nsarc)text_train, text_test, nast_train, nast_test = train_test_split(data.text, data.nastiness, test_size=0.2, random_state=42)# This will split your data into a train set (80%) and a test set (20%)# The random_state parameter ensures reproducibility of the split

In [ ]:
import torchfrom transformers import BertTokenizer, BertModel# Cargar el tokenizador y el modelo de BERTtokenizer = BertTokenizer.from_pretrained("dccuchile/bert-base-spanish-wwm-cased")model = BertModel.from_pretrained("dccuchile/bert-base-spanish-wwm-cased", output_hidden_states=True)# Pon el modelo en modo de evaluaciónmodel.eval()

In [ ]:
tokens_train = [tokenizer.encode_plus(texto, add_special_tokens=True, max_length=256, pad_to_max_length=True, truncation=True) for texto in text_train]tokens_test = [tokenizer.encode_plus(texto, add_special_tokens=True, max_length=256, pad_to_max_length=True, truncation=True) for texto in text_test]input_ids_train = [tokens['input_ids'] for tokens in tokens_train]attention_masks_train = [tokens['attention_mask'] for tokens in tokens_train]input_ids_test = [tokens['input_ids'] for tokens in tokens_test]attention_masks_test = [tokens['attention_mask'] for tokens in tokens_test]

In [ ]:
pos=1input_ids_test[pos].count(1),attention_masks_test[pos].count(0)

In [ ]:
train_inputs = torch.stack([torch.tensor(np.array(id)) for id in input_ids_train])train_masks = torch.stack([torch.tensor(np.array(mask)) for mask in attention_masks_train])test_inputs = torch.stack([torch.tensor(np.array(id)) for id in input_ids_test])test_masks = torch.stack([torch.tensor(np.array(mask)) for mask in attention_masks_test])

In [ ]:
train_inputs.size()

In [ ]:
train_data = TensorDataset(train_inputs,train_masks) train_dataloader = DataLoader(train_data,batch_size=32)test_data = TensorDataset(test_inputs,test_masks) test_dataloader = DataLoader(test_data,batch_size=32)

In [ ]:
device = torch.device("cpu")output_test_tensor = []for batch in tqdm(test_dataloader, desc="Processing"):  # Add batch to GPU  batch = tuple(t.to(device) for t in batch)  with torch.no_grad():    # Unpack the inputs from our dataloader    b_input_ids, b_input_mask = batch    output = model(b_input_ids, b_input_mask)    #print(len(output[0][0]))    #print(output)    output = output.last_hidden_state.detach().to('cpu')    output_test_tensor.extend(output)

In [ ]:
# Stack tensors along a new dimension (dim=0)output_test_tensor = torch.stack(output_test_tensor, dim=0)print(output_test_tensor.size())

In [ ]:
mean_test_tensor = torch.mean(output_test_tensor, dim=1)print(mean_test_tensor.size())

In [ ]:
#file_path = "./data/output_train_tensor_shuffled.pt"# Save the tensor to the file#torch.save(output_train_tensor, file_path)# Specify the file pathfile_path = "./data/mean_test_tensor_shuffled.pt"# Save the tensor to the filetorch.save(mean_test_tensor, file_path)

In [ ]:
import torchnast_train_tensor = torch.tensor(nast_train.values)nast_test_tensor = torch.tensor(nast_test.values)# Specify the file pathfile_path = "./data/nast_train_tensor_shuffled.pt"# Save the tensor to the filetorch.save(nast_train_tensor, file_path)# Specify the file pathfile_path = "./data/nast_test_tensor_shuffled.pt"# Save the tensor to the filetorch.save(nast_test_tensor, file_path)